### iStar

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
os.chdir(os.path.abspath(".."))
from utils.Super_resolution_enhancement_utils import ForwardSumModel

pattern = 'pixel_real'
Save_path = 'Result/Super_resolution_enhancement/skin'
Src_path = 'Dataset/Super_resolution_enhancement/skin'
with open(os.path.join(Src_path,'image_embedding_combine.pkl'), 'rb') as file:
    embs = pickle.load(file)
emb = embs['x']

with open(os.path.join(Save_path,f"istar_{pattern}_data.pkl"), "rb") as f:
    istar_data = pickle.load(f)
test_genes  = np.load(os.path.join(Src_path,f"genes_{pattern}_test.npy"), allow_pickle=True).tolist()

cnt_train = istar_data['train_Y']
cnt_test = istar_data['test_Y']
seen_genes = istar_data['seen_genes']
num_genes = len(seen_genes)
num_epochs = 400
batch_size = 128

predictions_set = {}
device = 'cuda'

with open(os.path.join(Src_path,'image_embedding_sectionB.pickle'), 'rb') as file:
    image_embs = pickle.load(file)
image_cls = np.stack(image_embs['cls'], axis=2)
image_sub = np.stack(image_embs['sub'], axis=2)
image_rgb = np.stack(image_embs['rgb'], axis=2)
image1 = np.concatenate((image_cls, image_sub, image_rgb),axis=-1)
px_num1, py_num1, n_features = image1.shape
image_flat1 = image1.reshape(px_num1*py_num1,n_features)
image_flat1 = torch.tensor(image1, dtype=torch.float32).to(device)
hyper_dataset1 = TensorDataset(image_flat1)
hyper_loader1 = DataLoader(hyper_dataset1, batch_size=32, shuffle=False)

with open(os.path.join(Src_path,'image_embedding_sectionA.pickle'), 'rb') as file:
    image_embs = pickle.load(file)
image_cls = np.stack(image_embs['cls'], axis=2)
image_sub = np.stack(image_embs['sub'], axis=2)
image_rgb = np.stack(image_embs['rgb'], axis=2)
image2 = np.concatenate((image_cls, image_sub, image_rgb),axis=-1)
px_num2, py_num2, n_features = image2.shape
image_flat2 = image2.reshape(px_num2*py_num2,n_features)
image_flat2 = torch.tensor(image2, dtype=torch.float32).to(device)
hyper_dataset2 = TensorDataset(image_flat2)
hyper_loader2 = DataLoader(hyper_dataset2, batch_size=32, shuffle=False)

'''
Training section A -> Testing section B
'''
expression_train_slide2 = torch.cat(cnt_train, dim=1).to(device)
expression_test_slide1 = torch.cat(cnt_test, dim=1).to(device)
emb_slide1 = torch.tensor(emb[:expression_test_slide1.shape[0],:,:], dtype=torch.float32).to(device) 
emb_slide2 = torch.tensor(emb[expression_test_slide1.shape[0]:,:,:], dtype=torch.float32).to(device) 
print(emb_slide1.shape,expression_test_slide1.shape,emb_slide2.shape,expression_train_slide2.shape)

train_dataset_slide2 = TensorDataset(emb_slide2, expression_train_slide2)
test_dataset_slide1 = TensorDataset(emb_slide1, expression_test_slide1)
train_loader_silde2 = DataLoader(train_dataset_slide2, batch_size=batch_size, shuffle=True)
test_loader_silde1 = DataLoader(test_dataset_slide1, batch_size=batch_size, shuffle=False)

model = ForwardSumModel(n_inp=emb_slide2.shape[-1], n_out=expression_train_slide2.shape[-1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

all_predictions = []
all_groundtruth = []
# Loss functions
criterion = nn.MSELoss()
# Training loop
model.train()
for epoch in tqdm(range(num_epochs), desc="Epochs", ncols=100, dynamic_ncols=True):
    total_loss = 0
    for batch in train_loader_silde2:
        optimizer.zero_grad()
        batch_X, batch_y = batch
        y_pred  = model(batch_X)
        y_mean_pred = y_pred.mean(-2)
        total_batch_loss = ((y_mean_pred - batch_y)**2).mean()
        total_batch_loss.backward()
        optimizer.step()       
        total_loss += total_batch_loss.item()        
    
    train_loss = total_loss / len(train_loader_silde2) 
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{num_epochs} | "
              f"Train Loss: {train_loss:.6f}")

model.eval()
with torch.no_grad():
    for batch in test_loader_silde1:
        batch_X, batch_y = batch
        y_pred  = model(batch_X)
        y_mean_pred = y_pred.mean(-2)
        all_predictions.append(y_mean_pred.cpu().numpy())
        all_groundtruth.append(batch_y.cpu().numpy())
    y_pred_final = np.concatenate(all_predictions).flatten(order='F') 
    y_true_final = np.concatenate(all_groundtruth).flatten(order='F') 
    
hyper_predictions = []
with torch.no_grad():        
    for batch in hyper_loader1:
        batch_X = batch[0]
        hyper_y_pred = model(batch_X).cpu().numpy()
        hyper_predictions.append(hyper_y_pred)
    predictions = np.concatenate(hyper_predictions, axis=0)

i = 0
for name in test_genes:
    predictions_set[name] = predictions[:, :, i]
    i = i+1
    
'''
save results
'''
with open(os.path.join(Save_path,f"iStar_{pattern}_hyper.pkl"), 'wb') as f:
        pickle.dump(predictions_set, f)   

np.save(os.path.join(Save_path,f"iStar_{pattern}_true.npy"),y_true_final)
np.save(os.path.join(Save_path,f"iStar_{pattern}_pred.npy"),y_pred_final)

torch.Size([2170, 37, 579]) torch.Size([2170, 98]) torch.Size([1209, 37, 579]) torch.Size([1209, 98])


Epochs:   1%|          | 3/400 [00:00<00:15, 24.93it/s]

Epoch 0/400 | Train Loss: 0.029359


Epochs:  26%|██▋       | 105/400 [00:04<00:11, 25.74it/s]

Epoch 100/400 | Train Loss: 0.007533


Epochs:  51%|█████     | 204/400 [00:07<00:07, 25.78it/s]

Epoch 200/400 | Train Loss: 0.005767


Epochs:  76%|███████▋  | 306/400 [00:11<00:03, 25.83it/s]

Epoch 300/400 | Train Loss: 0.004980


Epochs: 100%|██████████| 400/400 [00:15<00:00, 25.84it/s]
